# Patristic Text Archive (PTA) Demo

This notebook demonstrates `PTAReader` for working with the [Patristic Text Archive](https://pta.bbaw.de) —
a Berlin-Brandenburg Academy project providing open-access ancient Christian texts in Greek and Latin (CC-BY 4.0).

**Corpus (as of 2026-05-22):**

| Language | Text files | Approx. tokens |
|----------|-----------|----------------|
| Greek    | 213       | ~2.3M          |
| Latin    | 7         | ~20K           |

**Key features of `PTAReader`:**
- Yields one Doc per `<div type="textpart">` section — right-sized chunks for NLP
- Per-Doc metadata: CTS URN, language (`lat`/`grc`), author, title, div_type, div_n, citation
- Auto-download from GitHub on first use (`--depth 1` shallow clone)
- `<note>` elements stripped from body text by default
- Works with both Latin (`la_core_web_lg`) and Greek (`grc_dep_treebanks_trf`) models

In [1]:
from latincyreaders import PTAReader, AnnotationLevel
from pathlib import Path
from pprint import pprint
from collections import Counter, defaultdict

## Set up the reader

`PTAReader()` with no arguments auto-downloads the corpus into `~/latincy_data/pta_data`
on first use (shallow clone, ~50 MB).

Or pass a path to an existing checkout:

In [2]:
# Default location after auto-download (or PTAReader() to trigger download)
PTA_DATA = Path.home() / "latincy_data" / "pta_data" / "data"

# Survey reader — no NLP, just file discovery
reader = PTAReader(root=PTA_DATA, annotation_level=AnnotationLevel.NONE)

## File discovery

By default, `PTAReader` discovers all `*.pta-*.xml` files (excluding `__cts__.xml` metadata files).

In [3]:
all_files = reader.fileids()
lat_files = reader.fileids(match=r"\.pta-lat")
grc_files = reader.fileids(match=r"\.pta-grc")

print(f"Total files : {len(all_files)}")
print(f"Latin files : {len(lat_files)}")
print(f"Greek files : {len(grc_files)}")
print()
print("Latin files:")
pprint(lat_files)

Total files : 536
Latin files : 7
Greek files : 213

Latin files:
['pta0001/pta014/pta0001.pta014.pta-lat1.xml',
 'pta0001/pta028/pta0001.pta028.pta-lat1.xml',
 'pta0001/pta030/pta0001.pta030.pta-lat1.xml',
 'pta0001/pta031/pta0001.pta031.pta-lat1.xml',
 'pta0001/pta054/pta0001.pta054.pta-lat1.xml',
 'pta0001/pta063/pta0001.pta063.pta-lat1.xml',
 'pta0030/pta016/pta0030.pta016.pta-lat1.xml']


## Raw text extraction

`texts()` yields one string per `<div type="textpart">` section — zero NLP overhead.

### Latin

In [4]:
reader_lat = PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-lat*.xml",
    annotation_level=AnnotationLevel.NONE,
)

lat_texts = list(reader_lat.texts())
print(f"Latin sections: {len(lat_texts)}")
print()
print("First section (first 500 chars):")
print(lat_texts[0][:500])

Latin sections: 90

First section (first 500 chars):
Miratus sum vobiscum, ò Christi amantissimi, quod novae gratiae Evangelii per antiquum testamentum praedicatae sunt. Illuminatus sum vobiscum mente, quoniam vidi apostolicas notas in propheticis vocibus illustratas. Confiteor gratiam, ut accipiam mercedem. Vidi nova mysteria per vetus (testamentum) praedicata; quum Abraham juramentum servo per femur suum dabat, in femur manum mittere faciebat, et Deum caeli et terrae dictis testem advocabat. Pone, inquit, manum tuam sub femore meo, et adjurabo t


### Greek

In [5]:
reader_grc = PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-grc*.xml",
    annotation_level=AnnotationLevel.NONE,
)

# One file — count its sections
one_grc = reader_grc.fileids()[0]
grc_sections = list(reader_grc.texts(fileids=one_grc))
print(f"File: {one_grc}")
print(f"Sections: {len(grc_sections)}")
print()
print("Section 1 (first 300 chars):")
print(grc_sections[0][:300])

File: pta0001/pta001/pta0001.pta001.pta-grc1.xml
Sections: 11

Section 1 (first 300 chars):
Πᾶσα γραφὴ θεόπνευστος καὶ ὠφέλιμος , ἀρχὴν ἔχουσα καὶ πηγὴν τῆς εὐσεβείας τὸ πνεῦμα τῆς ἀληθείας . Ἀπὸ γὰρ τοῦ ἁγίου καὶ προσκυνητοῦ πνεύματος, ὥσπερ ἀπό τινος εὐθαλοῦς καὶ γονίμου πηγῆς, πάντα πηγάζει τὰ θεῖα νάματα. Καὶ ὅσα ὁ νόμος διαγορεύει, ὅσα προφῆται θεσπίζουσιν, ὅσα ἀπόστολοι κηρύττουσι, π


## Metadata

Every Doc chunk carries file-level metadata (URN, language, author, title)
plus per-section fields (div_type, div_n, citation).

In [6]:
reader_tok = PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-lat*.xml",
    annotation_level=AnnotationLevel.TOKENIZE,
)

# Metadata for the first Latin section
doc = next(reader_tok.docs())
pprint(doc._.metadata)

{'author': 'Severianus Gabalensis',
 'citation': '1',
 'div_n': '1',
 'div_type': 'section',
 'filename': 'pta0001.pta014.pta-lat1.xml',
 'language': 'lat',
 'path': '~/latincy_data/pta_data/data/pta0001/pta014/pta0001.pta014.pta-lat1.xml',
 'title': 'In illud: Pone manum tuam',
 'urn': 'urn:cts:pta:pta0001.pta014.pta-lat1'}


In [7]:
# Walk sections of one file — citation progression
first_lat = reader_tok.fileids()[0]
print(f"File: {first_lat}")
print()
for doc in reader_tok.docs(fileids=first_lat):
    m = doc._.metadata
    sents = list(doc.sents)
    print(f"  §{m['citation']:>3}  {m['div_type']:<10}  {len(sents):>3} sents  {len(doc):>5} tokens")

File: pta0001/pta014/pta0001.pta014.pta-lat1.xml



  §  1  section     336 sents   6772 tokens

## Corpus overview: works by language

CTS URNs follow the pattern `urn:cts:pta:{authorID}.{workID}.{lang}{version}`.

In [8]:
reader_all = PTAReader(
    root=PTA_DATA,
    annotation_level=AnnotationLevel.TOKENIZE,
)

work_sections: dict[str, list] = defaultdict(list)
work_meta: dict[str, dict] = {}

for doc in reader_all.docs():
    m = doc._.metadata
    urn = m.get("urn") or ""
    if not urn:
        continue
    work_sections[urn].append(m.get("citation", ""))
    if urn not in work_meta:
        work_meta[urn] = m

lat_works = {u: v for u, v in work_sections.items() if ".pta-lat" in u}
grc_works = {u: v for u, v in work_sections.items() if ".pta-grc" in u}

print(f"Latin works:  {len(lat_works)}")
print(f"Greek works:  {len(grc_works)}")
print(f"Total sections: {sum(len(v) for v in work_sections.values())}")

Latin works:  7
Greek works:  159
Total sections: 4926


In [9]:
# Latin works table
print("Latin works:")
print(f"  {'URN':<50} {'§§':>4}  Author: Title")
print("  " + "-" * 85)
for urn in sorted(lat_works):
    m = work_meta[urn]
    title = (m.get("title") or "")[:40]
    author = (m.get("author") or "?")[:25]
    print(f"  {urn:<50} {len(lat_works[urn]):>4}  {author}: {title}")

Latin works:
  URN                                                  §§  Author: Title
  -------------------------------------------------------------------------------------
  urn:cts:pta:pta0001.pta014.pta-lat1                   1  Severianus Gabalensis: In illud: Pone manum tuam
  urn:cts:pta:pta0001.pta028.pta-lat1                   3  Severianus Gabalensis (Au: In theophaniam
  urn:cts:pta:pta0001.pta030.pta-lat1                   3  Severianus Gabalensis: Homilia de pace
  urn:cts:pta:pta0001.pta031.pta-lat1                  32  Severianus
              : In illud: Pater, transeat a me calix ist
  urn:cts:pta:pta0001.pta054.pta-lat1                  18  Severianus Gabalensis: De cruce et latrone
  urn:cts:pta:pta0001.pta063.pta-lat1                  28  Severianus Gabalensis: De incarnatione
  urn:cts:pta:pta0030.pta016.pta-lat1                   5  Eusebius (»Alexandrinus«): De die dominica (Versio A)


In [10]:
# Greek works — first 20
print("Greek works (first 20):")
print(f"  {'URN':<50} {'§§':>4}  Author")
print("  " + "-" * 75)
for urn in sorted(grc_works)[:20]:
    m = work_meta[urn]
    author = (m.get("author") or "?")[:30]
    print(f"  {urn:<50} {len(grc_works[urn]):>4}  {author}")

Greek works (first 20):
  URN                                                  §§  Author
  ---------------------------------------------------------------------------
  urn:cts:pta:pta0001.pta001.pta-grc1                  11  Severianus Gabalensis
  urn:cts:pta:pta0001.pta001.pta-grcBibex              11  Severianus Gabalensis
  urn:cts:pta:pta0001.pta002.pta-grc1                   7  Severianus Gabalensis
  urn:cts:pta:pta0001.pta003.pta-grc1                  40  Severianus Gabalensis
  urn:cts:pta:pta0001.pta004.pta-grc1                  11  Severianus Gabalensis
  urn:cts:pta:pta0001.pta005.pta-grc1                   4  Severianus Gabalensis
  urn:cts:pta:pta0001.pta006.pta-grc1                   6  Severianus Gabalensis
  urn:cts:pta:pta0001.pta007.pta-grc1                   7  Severianus Gabalensis
  urn:cts:pta:pta0001.pta008.pta-grc1                   7  Severianus Gabalensis
  urn:cts:pta:pta0001.pta009.pta-grc1                   8  Severianus Gabalensis
  urn:cts:pta:pta0001.

## NLP: Latin with `la_core_web_lg`

Use `AnnotationLevel.TOKENIZE` for fast tokenization/sentence segmentation,
or `AnnotationLevel.FULL` for full parsing + lemmatization.

In [11]:
reader_lat_nlp = PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-lat*.xml",
    annotation_level=AnnotationLevel.TOKENIZE,
)

total_tokens = 0
total_sents  = 0

for doc in reader_lat_nlp.docs():
    m = doc._.metadata
    sents = list(doc.sents)
    total_tokens += len(doc)
    total_sents  += len(sents)
    print(f"§{m['citation']:>3}  {m['author'][:25]:<25}  {len(doc):>5} tokens  {len(sents):>3} sents")

print(f"\nTotal Latin: {total_tokens:,} tokens, {total_sents:,} sentences")

§  1  Severianus Gabalensis       6772 tokens  336 sents
§  1  Severianus Gabalensis (Au    190 tokens   12 sents
§  2  Severianus Gabalensis (Au    176 tokens    8 sents
§  3  Severianus Gabalensis (Au    164 tokens    7 sents
§  1  Severianus Gabalensis        386 tokens   19 sents
§  2  Severianus Gabalensis        216 tokens   10 sents
§  3  Severianus Gabalensis         86 tokens    3 sents
§  1  Severianus
                  155 tokens    6 sents
§  2  Severianus
                  141 tokens    3 sents
§  3  Severianus
                  430 tokens   19 sents
§  4  Severianus
                  264 tokens   16 sents
§  5  Severianus
                  281 tokens   14 sents
§  6  Severianus
                  231 tokens   12 sents
§  7  Severianus
                  204 tokens   10 sents
§  8  Severianus
                  246 tokens   14 sents
§  9  Severianus
                  343 tokens   15 sents
§ 10  Severianus
                  316 tokens   13 sents
§ 11  Severianus
              

In [12]:
# First 5 sentences from the first Latin section
first_doc = next(PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-lat*.xml",
    annotation_level=AnnotationLevel.TOKENIZE,
).docs())

m = first_doc._.metadata
print(f"{m['author']}, {m['title']}")
print(f"URN: {m['urn']}  §{m['citation']}")
print()
for i, sent in enumerate(list(first_doc.sents)[:5]):
    print(f"[{i+1}] {sent.text[:130]}")

Severianus Gabalensis, In illud: Pone manum tuam
URN: urn:cts:pta:pta0001.pta014.pta-lat1  §1

[1] Miratus sum vobiscum, ò Christi amantissimi, quod novae gratiae Evangelii per antiquum testamentum praedicatae sunt.
[2] Illuminatus sum vobiscum mente, quoniam vidi apostolicas notas in propheticis vocibus illustratas.
[3] Confiteor gratiam, ut accipiam mercedem.
[4] Vidi nova mysteria per vetus (testamentum) praedicata; quum Abraham juramentum servo per femur suum dabat, in femur manum mittere 
[5] Pone, inquit, manum tuam sub femore meo, et adjurabo te in Dominum Deum creatorem caeli et terrae.


## NLP: Greek with `grc_dep_treebanks_trf`

Install the Greek model once:
```bash
pip install https://huggingface.co/chcaa/grc_dep_treebanks_trf/resolve/main/grc_dep_treebanks_trf-any-py3-none-any.whl
```

In [13]:
try:
    grc_reader = PTAReader(
        root=PTA_DATA,
        fileids=reader_grc.fileids()[0],   # first Greek file
        annotation_level=AnnotationLevel.TOKENIZE,
        model_name="grc_dep_treebanks_trf",
        lang="grc",
    )

    grc_total_tokens = 0
    grc_total_sents  = 0
    for doc in grc_reader.docs():
        m = doc._.metadata
        sents = list(doc.sents)
        grc_total_tokens += len(doc)
        grc_total_sents  += len(sents)

    print(f"File: {grc_reader.fileids()[0]}")
    print(f"Total: {grc_total_tokens:,} tokens, {grc_total_sents:,} sentences")
    print()

    # Show first 5 sentences from first section
    first_grc = next(PTAReader(
        root=PTA_DATA,
        fileids=reader_grc.fileids()[0],
        annotation_level=AnnotationLevel.TOKENIZE,
        model_name="grc_dep_treebanks_trf",
        lang="grc",
    ).docs())
    m = first_grc._.metadata
    print(f"{m['author']}, {m['title']}")
    print(f"URN: {m['urn']}  §{m['citation']}")
    print()
    for i, sent in enumerate(list(first_grc.sents)[:5]):
        print(f"[{i+1}] {sent.text[:130]}")

except OSError:
    print("Greek model not installed.")
    print("Install: pip install https://huggingface.co/chcaa/grc_dep_treebanks_trf/resolve/main/grc_dep_treebanks_trf-any-py3-none-any.whl")
    print()
    # Raw Greek text still works without a model
    grc_raw = next(PTAReader(
        root=PTA_DATA,
        fileids=reader_grc.fileids()[0],
        annotation_level=AnnotationLevel.NONE,
    ).texts())
    print("Raw Greek text (first 300 chars):")
    print(grc_raw[:300])

File: pta0001/pta001/pta0001.pta001.pta-grc1.xml
Total: 4,553 tokens, 335 sentences

Severianus Gabalensis, De fide et lege naturae
URN: urn:cts:pta:pta0001.pta001.pta-grc1  §1

[1] Πᾶσα γραφὴ θεόπνευστος καὶ ὠφέλιμος , ἀρχὴν ἔχουσα καὶ πηγὴν τῆς εὐσεβείας τὸ πνεῦμα τῆς ἀληθείας .
[2] Ἀπὸ γὰρ τοῦ ἁγίου καὶ προσκυνητοῦ πνεύματος, ὥσπερ ἀπό τινος εὐθαλοῦς καὶ γονίμου πηγῆς, πάντα πηγάζει τὰ θεῖα νάματα.
[3] Καὶ ὅσα ὁ νόμος διαγορεύει, ὅσα προφῆται θεσπίζουσιν, ὅσα ἀπόστολοι κηρύττουσι, πάντα ταῦτα τῷ ἁγίῳ πνεύματι ἀληθινῶς ἀπεικάζεται 
[4] Πάντα γὰρ ἐνεργεῖ τὸ ἓν καὶ τὸ αὐτὸ πνεῦμα, διαιροῦν ἰδίᾳ ἑκάστῳ, καθὼς βούλεται.
[5] Διὰ τοῦτο πάντα ἀπαστράπτει τὰ κάλλη τῆς εὐσεβείας, καὶ λάμπουσιν οἱ λόγοι τῆς ἀληθείας, καὶ βρύουσι θησαυροὶ τῆς ἐνθέου σοφίας·


## Searching with `find_sents()`

### Latin: find sentences mentioning Abraham

In [14]:
reader_search_lat = PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-lat*.xml",
    annotation_level=AnnotationLevel.TOKENIZE,
)

hits = list(reader_search_lat.find_sents(pattern=r"\bAbraham\b"))
print(f"Sentences mentioning 'Abraham' (Latin): {len(hits)}")
print()
for hit in hits[:3]:
    print(f"  {hit['citation']}: {hit['sentence'][:120]}")
    print()

Sentences mentioning 'Abraham' (Latin): 8

  pta0001/pta014/pta0001.pta014.pta-lat1.xml:sent3: Vidi nova mysteria per vetus (testamentum) praedicata; quum Abraham juramentum servo per femur suum dabat, in femur manu

  pta0001/pta014/pta0001.pta014.pta-lat1.xml:sent8: Patet quod Abraham habebat in seipso semen salutare: de quo Paulus dicebat, Abrahae data est promissio, et semini ejus; 

  pta0001/pta014/pta0001.pta014.pta-lat1.xml:sent78: Ipse quoque Servator dum Judaeos alloquebatur, dixit: Amen amen dico vobis, antequam Abraham fieret, ego sum.



### Greek: find sentences containing a word

In [15]:
# Greek raw-text search — works without a Greek NLP model
# (uses AnnotationLevel.TOKENIZE with a blank Greek tokenizer)
reader_search_grc = PTAReader(
    root=PTA_DATA,
    fileids=reader_grc.fileids()[0],  # one file for speed
    annotation_level=AnnotationLevel.TOKENIZE,
    lang="grc",
)

hits_grc = list(reader_search_grc.find_sents(pattern=r"Ἀβραάμ"))
print(f"Sentences with 'Ἀβραάμ' (Greek): {len(hits_grc)}")
print()
for hit in hits_grc[:3]:
    print(f"  {hit['citation']}: {hit['sentence'][:130]}")
    print()

Sentences with 'Ἀβραάμ' (Greek): 0



## Word frequency: Latin content words

In [16]:
reader_freq = PTAReader(
    root=PTA_DATA,
    fileids="**/*.pta-lat*.xml",
    annotation_level=AnnotationLevel.TOKENIZE,
)

freq: Counter = Counter()
for tok in reader_freq.tokens():
    if tok.is_alpha and not tok.is_stop:
        freq[tok.lower_] += 1

print(f"Unique word forms: {len(freq):,}")
print()
print("Top 25 content words:")
for word, count in freq.most_common(25):
    print(f"  {word:<22} {count:>5}")

Unique word forms: 6,229

Top 25 content words:
  quod                     194
  ut                       147
  sunt                      84
  dei                       82
  sicut                     80
  erat                      76
  a                         72
  quid                      71
  me                        70
  deus                      67
  ait                       63
  sit                       62
  propter                   57
  haec                      55
  se                        53
  pater                     53
  quum                      48
  deum                      45
  dicit                     45
  te                        43
  dixit                     43
  tibi                      41
  verbum                    40
  verba                     40
  dominus                   39


## Greek character frequency (no model needed)

In [17]:
# Count Greek word forms across all sections of one work
# Uses AnnotationLevel.NONE (zero overhead) and simple string splitting
reader_grc_freq = PTAReader(
    root=PTA_DATA,
    fileids=reader_grc.fileids()[0],
    annotation_level=AnnotationLevel.NONE,
)

import re
grc_word_freq: Counter = Counter()
for text in reader_grc_freq.texts():
    for word in re.findall(r"[\u0370-\u03ff\u1f00-\u1fff]+", text.lower()):
        grc_word_freq[word] += 1

print(f"Unique Greek word forms (one work): {len(grc_word_freq):,}")
print()
print("Top 20 Greek word forms:")
for word, count in grc_word_freq.most_common(20):
    print(f"  {word:<25} {count:>4}")

Unique Greek word forms (one work): 1,514

Top 20 Greek word forms:
  καὶ                        224
  τὴν                         80
  ὁ                           78
  τὸ                          63
  τοῦ                         63
  τῆς                         61
  τῷ                          49
  ἡ                           49
  τῶν                         46
  τὰ                          40
  δὲ                          40
  γὰρ                         39
  τὸν                         39
  οὐκ                         32
  εἰς                         31
  ὡς                          31
  ἐν                          30
  μὲν                         29
  θεοῦ                        28
  τοῖς                        27
